In [ ]:
# Setup and Imports
import os
import json
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import f1_score, r2_score
from google.colab import drive

print("Mounting Google Drive...")
drive.mount('/content/drive')

GDRIVE_PATH = '/content/drive/MyDrive/'
BASE_DIR = os.path.join(GDRIVE_PATH, 'Baseline_Test/')
DATA_DIR = os.path.join(GDRIVE_PATH, 'FinalData/')
SEED = 42
os.makedirs(BASE_DIR, exist_ok=True)
print("Setup complete.")

In [ ]:

# Baseline Training Function
def train_baseline_models(dataset_name, df, task_type, text_col, label_cols):
    """Trains multiple baseline models, evaluates them, and saves the best one."""
    print(f"\n--- Starting Baseline Process for: {dataset_name} ---")


    dataset_folder = os.path.join(BASE_DIR, dataset_name)
    metrics_folder = os.path.join(dataset_folder, 'metrics')
    model_folder = os.path.join(dataset_folder, 'model')
    os.makedirs(metrics_folder, exist_ok=True)
    os.makedirs(model_folder, exist_ok=True)


    X = df[text_col]
    y = df[label_cols]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
    vectorizer = TfidfVectorizer(max_features=5000)
    X_train_tfidf = vectorizer.fit_transform(X_train)
    X_test_tfidf = vectorizer.transform(X_test)


    if task_type == 'classification':
        models = {
            'Naive Bayes': OneVsRestClassifier(MultinomialNB()),
            'Logistic Regression': OneVsRestClassifier(LogisticRegression(random_state=SEED, solver='liblinear')),
            'Linear SVM': OneVsRestClassifier(LinearSVC(random_state=SEED, dual=False, max_iter=2000))
        }
        metric_func = f1_score
        metric_name = 'f1_weighted'
        metric_args = {'average': 'weighted', 'zero_division': 0}
    else:
        models = { 'Ridge Regression': Ridge(random_state=SEED) }
        metric_func = r2_score
        metric_name = 'r2_score'
        metric_args = {}


    results = {}
    best_score = -np.inf
    best_model_name = ""
    best_model_obj = None

    for name, model in models.items():
        print(f"  -> Training {name}...")
        model.fit(X_train_tfidf, y_train)
        preds = model.predict(X_test_tfidf)
        score = metric_func(y_test, preds, **metric_args)
        results[name] = score
        print(f"     - Score ({metric_name}): {score:.4f}")
        if score > best_score:
            best_score = score
            best_model_name = name
            best_model_obj = model


    metrics_path = os.path.join(metrics_folder, 'baseline_metrics.json')
    with open(metrics_path, 'w') as f:
        json.dump(results, f, indent=4)
    print(f" Metrics saved to: {metrics_path}")

    model_path = os.path.join(model_folder, f'best_model_{best_model_name.replace(" ", "_").lower()}.pkl')
    vectorizer_path = os.path.join(model_folder, 'tfidf_vectorizer.pkl')
    joblib.dump(best_model_obj, model_path)
    joblib.dump(vectorizer, vectorizer_path)
    print(f" Best model ({best_model_name}) and vectorizer saved to: {model_folder}")

print(" Master baseline training function defined.")

In [ ]:

# Essaysbig5
print("\n--- Training Baselines for Essaysbig5 ---")
df_essays = pd.read_csv(os.path.join(DATA_DIR, 'essaysbig5_clean.csv')).dropna()
train_baseline_models('Essaysbig5', df_essays, 'classification', 'text', ['O', 'C', 'E', 'A', 'N'])

In [ ]:
# GoEmotions
print("\n--- Training Baselines for GoEmotions ---")
from sklearn.preprocessing import MultiLabelBinarizer
df_go = pd.read_csv(os.path.join(DATA_DIR, 'goemotions_clean.csv')).dropna()
mlb = MultiLabelBinarizer()
y_go_labels = mlb.fit_transform(df_go['emotions'].str.split(','))
df_go_transformed = pd.concat([df_go.reset_index(drop=True), pd.DataFrame(y_go_labels, columns=mlb.classes_)], axis=1)
train_baseline_models('GoEmotions', df_go_transformed, 'classification', 'text', mlb.classes_.tolist())

In [ ]:
# Pandora
print("\n--- Training Baselines for Pandora ---")
df_pandora = pd.read_csv(os.path.join(DATA_DIR, 'pandora_clean.csv')).dropna()
train_baseline_models('Pandora', df_pandora, 'regression', 'text', ['agreeableness', 'openness', 'conscientiousness', 'extraversion', 'neuroticism'])


In [ ]:
# EmoBank
print("\n--- Training Baselines for EmoBank ---")
df_emobank = pd.read_csv(os.path.join(DATA_DIR, 'emobank_clean.csv')).dropna()
train_baseline_models('EmoBank', df_emobank, 'regression', 'text', ['V', 'A', 'D'])